# 🧬 RNA → Protein: 유전 정보의 흐름
### Claude 활용 단백체 분석 실습 — 보너스 강의  
**KBSI · 황희연 · 2026**

---
이 노트북에서는 **mRNA 서열로부터 단백질 서열을 직접 해독**하는 과정을 Python으로 구현합니다.  
별도 설치 없이 순수 Python으로 실행됩니다.

### 학습 목표
1. Central Dogma (DNA → RNA → Protein) 이해  
2. 코돈 테이블을 Python 딕셔너리로 구현  
3. mRNA 번역(Translation) 함수 작성  
4. Open Reading Frame(ORF) 찾기  
5. 실제 인간 인슐린 서열로 검증  
6. 트립신 분해와 질량분석 연결


---
## 1. 🔬 Central Dogma — 유전 정보의 중심 원리

```
DNA  →  (전사, Transcription)  →  mRNA  →  (번역, Translation)  →  Protein
```

| | DNA | mRNA | Protein |
|--|-----|------|---------|
| 구성 단위 | 뉴클레오타이드 | 뉴클레오타이드 | 아미노산 |
| 염기 | A, T, G, C | A, **U**, G, C | 20종 |
| 가닥 | 이중 | 단일 | 3D 구조 |

### DNA → mRNA: T(티민) → U(우라실)


In [ ]:
# DNA → mRNA 전사
def dna_to_mrna(dna):
    return dna.upper().replace('T', 'U')

dna = "ATGAAAGCTGCTGGCTTCGAATAA"
mrna = dna_to_mrna(dna)
print(f"DNA  : {dna}")
print(f"mRNA : {mrna}")
print()
# 코돈 단위 시각화
codons = [mrna[i:i+3] for i in range(0, len(mrna), 3)]
print("코돈 :", " | ".join(codons))


---
## 2. 📖 코돈 테이블 (Codon Table)

- 4개 염기 × 3자리 = **64개** 코돈
- **20개** 아미노산으로 대응 (중복 코드)
- **시작 코돈**: AUG → 메티오닌(M)
- **종지 코돈**: UAA, UAG, UGA → 번역 종료 (`*`)


In [ ]:
CODON_TABLE = {
    'UUU':'F','UUC':'F','UUA':'L','UUG':'L',
    'CUU':'L','CUC':'L','CUA':'L','CUG':'L',
    'AUU':'I','AUC':'I','AUA':'I','AUG':'M',
    'GUU':'V','GUC':'V','GUA':'V','GUG':'V',
    'UCU':'S','UCC':'S','UCA':'S','UCG':'S',
    'CCU':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACU':'T','ACC':'T','ACA':'T','ACG':'T',
    'GCU':'A','GCC':'A','GCA':'A','GCG':'A',
    'UAU':'Y','UAC':'Y','UAA':'*','UAG':'*',
    'CAU':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAU':'N','AAC':'N','AAA':'K','AAG':'K',
    'GAU':'D','GAC':'D','GAA':'E','GAG':'E',
    'UGU':'C','UGC':'C','UGA':'*','UGG':'W',
    'CGU':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGU':'S','AGC':'S','AGA':'R','AGG':'R',
    'GGU':'G','GGC':'G','GGA':'G','GGG':'G',
}

print(f"총 코돈 수: {len(CODON_TABLE)}개")
stops = [k for k,v in CODON_TABLE.items() if v == '*']
print(f"시작 코돈: AUG → M (메티오닌)")
print(f"종지 코돈: {stops}")
print()

# 아미노산 전체 이름
AA_INFO = {
    'A':('Ala','알라닌'),    'R':('Arg','아르기닌'),
    'N':('Asn','아스파라긴'),'D':('Asp','아스파라긴산'),
    'C':('Cys','시스테인'),  'Q':('Gln','글루타민'),
    'E':('Glu','글루탐산'),  'G':('Gly','글리신'),
    'H':('His','히스티딘'),  'I':('Ile','이소류신'),
    'L':('Leu','류신'),      'K':('Lys','라이신'),
    'M':('Met','메티오닌'),  'F':('Phe','페닐알라닌'),
    'P':('Pro','프롤린'),    'S':('Ser','세린'),
    'T':('Thr','트레오닌'),  'W':('Trp','트립토판'),
    'Y':('Tyr','티로신'),    'V':('Val','발린'),
}

# 특정 코돈 예시
print("코돈 조회 예시:")
for codon in ['AUG','UAA','UGU','GGG','AAA']:
    aa = CODON_TABLE[codon]
    info = AA_INFO.get(aa, ('Stop','종지'))
    print(f"  {codon} → {aa} ({info[0]}, {info[1]})")


---
## 3. 🔄 mRNA → Protein 번역 함수

번역 과정:
1. AUG (시작 코돈) 찾기
2. 3개씩 읽어서 아미노산으로 변환
3. 종지 코돈 (UAA/UAG/UGA) 만나면 중단


In [ ]:
def translate(seq, start_from_aug=False):
    '''mRNA(또는 DNA) 서열을 단백질로 번역'''
    seq = seq.upper().replace('T', 'U')  # DNA → mRNA 자동 변환
    
    # 시작 코돈(AUG)부터 번역 옵션
    if start_from_aug:
        pos = seq.find('AUG')
        if pos == -1:
            return ''
        seq = seq[pos:]
    
    protein = []
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i+3]
        if len(codon) < 3:
            break
        aa = CODON_TABLE.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    
    return ''.join(protein)


# ── 단계별 시각화 ──────────────────────────────
mrna = "AUGAAAGCUUGUUAA"
print(f"mRNA: {mrna}")
print()
print(f"{'코돈':^6} {'아미노산':^6} {'역할':^8}")
print("-" * 25)
for i in range(0, len(mrna)-2, 3):
    codon = mrna[i:i+3]
    aa = CODON_TABLE.get(codon, 'X')
    role = '← 시작' if codon=='AUG' else '← 종지' if aa=='*' else ''
    print(f"  {codon}   {aa:^6}  {role}")

print()
result = translate(mrna)
print(f"단백질: {result}  ({len(result)} aa)")


In [ ]:
# ── 실습: 여러 서열 번역해보기 ──────────────────
sequences = {
    "짧은 펩타이드":  "AUGGGCUGUUAA",
    "중간 서열":       "AUGAAAGCUGCUGGCUUCGAAAUGGUGACUUAA",
    "AUG 없는 서열":   "GCUAAAGCUUAA",    # start_from_aug=False → 번역 안 됨
}

for name, seq in sequences.items():
    protein = translate(seq)
    print(f"{name}")
    print(f"  입력: {seq}")
    print(f"  결과: {protein if protein else '(번역 불가 — AUG 없음)'}")
    print()


---
## 4. 🔍 ORF (Open Reading Frame) 찾기

**Reading Frame** = 어느 위치부터 3개씩 읽느냐 (0, 1, 2번 위치 중 하나)

```
AUGCAUGCUAG
AUG|CAU|GCU|AG    ← Frame 1 (0번째부터)
 A|UGC|AUG|CUA|G  ← Frame 2 (1번째부터)
 AU|GCA|UGC|UAG   ← Frame 3 (2번째부터)
```

**ORF** = AUG(시작) ~ 종지코돈 까지의 번역 가능 구간


In [ ]:
def find_orfs(seq, min_aa=5):
    '''서열에서 모든 ORF 찾기 (정방향 3 frames)'''
    seq = seq.upper().replace('T', 'U')
    orfs = []
    
    for frame in range(3):
        i = frame
        while i <= len(seq) - 3:
            if seq[i:i+3] == 'AUG':          # 시작 코돈
                protein = []
                j = i
                while j <= len(seq) - 3:
                    codon = seq[j:j+3]
                    aa = CODON_TABLE.get(codon, 'X')
                    if aa == '*':             # 종지 코돈
                        if len(protein) >= min_aa:
                            orfs.append({
                                'frame':   frame + 1,
                                'start':   i + 1,    # 1-based
                                'end':     j + 3,
                                'length':  len(protein),
                                'protein': ''.join(protein)
                            })
                        break
                    protein.append(aa)
                    j += 3
                i = j + 3
            else:
                i += 3
    
    return sorted(orfs, key=lambda x: -x['length'])


# ── 테스트 ──────────────────────────────────────
test = "GCAUGAAAGCUGUUAAGGCUAUGCUUAAAGCUUAA"
orfs = find_orfs(test, min_aa=2)

print(f"서열: {test}")
print(f"총 {len(orfs)}개 ORF 발견\n")

for i, orf in enumerate(orfs, 1):
    print(f"ORF #{i} | Frame {orf['frame']} | {orf['start']}~{orf['end']}번 위치")
    print(f"  단백질: {orf['protein']}  ({orf['length']} aa)")


---
## 5. 🧪 실제 예제 — 인간 인슐린 (Insulin, INS)

- **유전자**: INS  
- **단백질**: Proinsulin → Insulin  
- **UniProt**: P01308  
- **mRNA**: NM_000207 (NCBI)

인슐린은 혈당 조절 호르몬으로, 질량분석 단백체 분석에서 자주 검출됩니다.


In [ ]:
# 인간 인슐린 CDS (NM_000207.3 기반)
INSULIN_CDS = (
    "ATGCCCTGTGGATGCGCCTCCTGCCCCTGCTGGCGCTGCTGGCCCTCTGGGGACCTGACC"
    "CAGCCGCAGCCTTTGTGAACCAACACCTGTGCGGCTCACACCTGGTGGAAGCTCTCTACC"
    "TAGTGTGCGGGGAACGAGGCTTCTTCTACACACCCAAGACCCGCCGGGAGGCAGAGGACC"
    "TGCAGGTGGGGCAGGTGGAGCTGGGCGGGGGCCCTGGTGCAGGCAGCCTGCAGCCCTTGG"
    "CCCTGGAGGGGTCCCTGCAGAAGCGTGGCATTGTGGAACAATGCTGTACCAGCATCTGCT"
    "CCCTCTACCAGCTGGAGAACTACTGCAACTAG"
)

# DNA → mRNA → Protein
insulin_mrna    = dna_to_mrna(INSULIN_CDS)
proinsulin_pred = translate(insulin_mrna)

print(f"CDS 길이:  {len(INSULIN_CDS)} nt")
print(f"단백질 길이: {len(proinsulin_pred)} aa")
print()
print(f"번역 결과 (Proinsulin):")
print(f"  {proinsulin_pred}")
print()

# 처음 10개 아미노산 상세
print("처음 10개 아미노산:")
for i, aa in enumerate(proinsulin_pred[:10]):
    three, kor = AA_INFO.get(aa, ('???','?'))
    print(f"  {i+1:>2}번: {aa}  {three}  {kor}")


In [ ]:
# UniProt P01308 서열과 비교
PROINSULIN_REF = "MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN"

print("=" * 55)
print("번역 결과 vs UniProt P01308 비교")
print("=" * 55)
print(f"번역 결과  ({len(proinsulin_pred):>3}aa): {proinsulin_pred[:25]}...")
print(f"UniProt    ({len(PROINSULIN_REF):>3}aa): {PROINSULIN_REF[:25]}...")
print()

# 일치율 계산
n = min(len(proinsulin_pred), len(PROINSULIN_REF))
matches = sum(a==b for a,b in zip(proinsulin_pred[:n], PROINSULIN_REF[:n]))
print(f"처음 {n}개 중 {matches}개 일치 ({matches/n*100:.1f}%)")
print()
print("💡 길이 차이 이유:")
print("  UniProt 서열은 Preproinsulin 전체")
print("  신호 펩타이드(Signal Peptide) + Proinsulin")
print("  성숙 인슐린은 B-C-A 체인으로 추가 절단됨")


---
## 6. 🔁 역상보 서열과 6 Reading Frames

실제 게놈 DNA는 이중 가닥입니다. 유전자는 어느 가닥에도 있을 수 있어서  
**6개** 리딩 프레임 전체를 탐색해야 합니다.

```
5' ──► 정방향 Frame 1, 2, 3
       역방향 Frame 1, 2, 3 ◄── 5' (역상보 가닥)
```


In [ ]:
def reverse_complement(seq):
    '''DNA/RNA 서열의 역상보 서열'''
    comp = {'A':'T','T':'A','G':'C','C':'G','U':'A','N':'N'}
    return ''.join(comp.get(b,'N') for b in reversed(seq.upper()))


def find_all_orfs(seq, min_aa=5):
    '''6개 리딩 프레임에서 모든 ORF 탐색'''
    seq = seq.upper().replace('U','T')
    rc  = reverse_complement(seq)
    all_orfs = []
    
    for strand, s in [('+', seq), ('-', rc)]:
        mrna = s.replace('T','U')
        orfs = find_orfs(mrna, min_aa=min_aa)
        for orf in orfs:
            orf['strand'] = strand
            all_orfs.append(orf)
    
    return sorted(all_orfs, key=lambda x: -x['length'])


# ── 예제 ──────────────────────────────────────
test_dna = "GCTAGCATGAAAGCTGCTGGCTTCGAAATGGTGACTGCTAAAGCTTAAGCTAGCATGCTTAAAGCTTAA"

all_orfs = find_all_orfs(test_dna, min_aa=3)

print(f"DNA 길이: {len(test_dna)} bp")
print(f"발견된 ORF: {len(all_orfs)}개\n")
print(f"{'가닥':^4}  {'Frame':^5}  {'위치':^10}  {'길이':^4}  단백질")
print("-" * 60)
for orf in all_orfs:
    print(f"  {orf['strand']:^4}   {orf['frame']:^5}  "
          f"{orf['start']:>4}~{orf['end']:<4}   {orf['length']:^4}  {orf['protein']}")


---
## 7. 🔬 트립신 분해와 질량분석 연결

질량분석(MS)에서는 단백질을 **트립신(Trypsin)**으로 잘라서 펩타이드를 분석합니다.
- 절단 위치: **K(라이신)** 또는 **R(아르기닌)** 뒤
- 예외: 다음 잔기가 **P(프롤린)**이면 절단하지 않음


In [ ]:
# 아미노산 단분자 질량 (Da)
AA_MASS = {
    'A':71.04,'R':156.10,'N':114.04,'D':115.03,'C':103.01,
    'Q':128.06,'E':129.04,'G':57.02, 'H':137.06,'I':113.08,
    'L':113.08,'K':128.09,'M':131.04,'F':147.07,'P':97.05,
    'S':87.03, 'T':101.05,'W':186.08,'Y':163.06,'V':99.07,
}

def calc_mass(peptide):
    return sum(AA_MASS.get(aa, 0) for aa in peptide) + 18.01  # +H2O

def trypsin_digest(protein, missed_cleavages=0):
    '''트립신 분해 시뮬레이션'''
    seq = protein.upper()
    sites = [0]
    for i in range(len(seq)-1):
        if seq[i] in ('K','R') and seq[i+1] != 'P':
            sites.append(i+1)
    sites.append(len(seq))
    
    base = []
    for i in range(len(sites)-1):
        pep = seq[sites[i]:sites[i+1]]
        if len(pep) >= 6:
            base.append((pep, sites[i]+1, sites[i+1]))
    
    result = list(base)
    for mc in range(1, missed_cleavages+1):
        for i in range(len(base)-mc):
            merged = ''.join(p[0] for p in base[i:i+mc+1])
            result.append((merged, base[i][1], base[i+mc][2]))
    
    return sorted(result, key=lambda x: x[1])


# ── 인슐린 트립신 분해 ──────────────────────────
peps = trypsin_digest(proinsulin_pred, missed_cleavages=1)

print(f"인슐린 Proinsulin 트립신 분해 (MC=1)")
print(f"{'#':>2}  {'펩타이드':^35}  {'위치':^10}  {'질량(Da)':>10}")
print("-" * 65)
for i, (pep, s, e) in enumerate(peps[:10], 1):
    mass = calc_mass(pep)
    print(f"{i:>2}  {pep:<35}  {s:>4}~{e:<4}   {mass:>10.2f}")


---
## 8. ✏️ 실습 문제

아래 문제들을 직접 풀어보세요!


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 문제 1: 아래 mRNA를 번역하세요
# 힌트: 유명한 단백질의 일부입니다
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
mystery_mrna = (
    "AUGGUUCACUUGACUCUGAGGAGAAAGUCAACGUGGUGGUGGCCUGCACCAGCAUGCAC"
    "UUGACACAGAGGAGCAGCAGCAGCAGCAGCAGCAGCAGUAA"
)

# 여기에 코드 작성 ↓
# protein1 = translate(???)
# print(protein1)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 문제 2: 아래 DNA에서 가장 긴 ORF를 찾으세요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
mystery_dna = (
    "GCTAGCTAGCATGAAAGCTGCTGGCTTCGAAATGGTGACTGCTAAAGCTTAA"
    "GCTAGCATGCTTAAAGCTGCTGGCTTCGAAATGGTGACTGCTTAAGCTAGC"
    "ATGGCTAGCATGAAAGCTTAAGC"
)

# 여기에 코드 작성 ↓
# orfs = find_all_orfs(???)
# print(orfs[0])  # 가장 긴 ORF


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 문제 3: BRCA1 단백질 일부를 트립신 분해하고
#         분자량 2000~3000 Da 펩타이드를 찾으세요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
brca1 = "MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLVEIRPIHKGLGIVHRDSKKAYLMDFHGLSTEDLHPTTK"

# 여기에 코드 작성 ↓
# peps = trypsin_digest(???)
# target = [... if 2000 <= calc_mass(pep) <= 3000]


---
## 9. 💡 정답

In [ ]:
print("=== 정답 1 ===")
p1 = translate(mystery_mrna)
print(f"단백질: {p1}")
print(f"길이:   {len(p1)} aa")
print()

print("=== 정답 2 ===")
orfs2 = find_all_orfs(mystery_dna, min_aa=3)
if orfs2:
    best = orfs2[0]
    print(f"가장 긴 ORF: {best['protein']} ({best['length']} aa)")
    print(f"가닥: {best['strand']}, Frame: {best['frame']}")
print()

print("=== 정답 3 ===")
brca1 = "MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLVEIRPIHKGLGIVHRDSKKAYLMDFHGLSTEDLHPTTK"
peps3 = trypsin_digest(brca1, missed_cleavages=1)
target = [(pep,s,e) for pep,s,e in peps3 if 2000 <= calc_mass(pep) <= 3000]
print(f"2000~3000 Da 범위: {len(target)}개")
for pep,s,e in target:
    print(f"  {pep}  ({calc_mass(pep):.2f} Da)")


---
## 10. 📚 정리

| 개념 | 핵심 포인트 |
|------|-----------|
| **Central Dogma** | DNA → mRNA → Protein |
| **전사** | T(티민) → U(우라실) 치환 |
| **코돈** | 3개 염기 = 1개 아미노산, 64개 코돈 |
| **시작 코돈** | AUG → 메티오닌(M) |
| **종지 코돈** | UAA, UAG, UGA |
| **ORF** | AUG ~ 종지코돈 구간 |
| **6 Reading Frames** | 정방향 3 + 역방향 3 |
| **트립신 분해** | K/R 뒤 절단 (P 제외) |

### BioPython으로 더 간편하게
```python
from Bio.Seq import Seq

seq = Seq("ATGAAAGCTGCTGGCTTCGAATAA")
mrna    = seq.transcribe()       # DNA → mRNA
protein = mrna.translate()       # mRNA → Protein
print(protein)  # MKAA*
```

### 다음 단계
- NCBI Entrez API로 실제 mRNA 서열 자동 다운로드  
- UniProt REST API로 단백질 정보 조회  
- 트립신 분해 펩타이드 → 질량분석 데이터와 비교
